In [0]:
%sql
-- 1. Aktualizacja zamówienia z tego samego dnia co initial load (po ostatnim ładowaniu)
UPDATE datamodeling.source.source_data
SET quantity = 2,
    last_updated = TIMESTAMP '2024-07-01 18:30:00'
WHERE order_id = 1001;

In [0]:
%sql
-- 2–4. Nowe zamówienia (MERGE = ponowne uruchomienie nie zrobi duplikatów)
MERGE INTO datamodeling.source.source_data t
USING (
    SELECT * FROM VALUES
      -- nowy klient, nowy kraj
      (1004, DATE '2024-07-02', 4, 'David Wilson', 'david@gmail.com',
       504, 'Samsung Galaxy S24', 'Electronics', 1, CAST(899.99 AS DECIMAL(10,2)),
       'PayPal', 'UK', TIMESTAMP '2024-07-02 09:20:00'),
      -- istniejący klient z nowym e-mailem (SCD1) i bez metody płatności (Unknown)
      (1005, DATE '2024-07-02', 2, 'Bob Smith', 'bob.smith@gmail.com',
       502, 'AirPods Pro', 'Electronics', 1, CAST(199.99 AS DECIMAL(10,2)),
       CAST(NULL AS STRING), 'USA', TIMESTAMP '2024-07-02 12:00:00')
    AS v(order_id, order_date, customer_id, customer_name, customer_email,
         product_id, product_name, product_category, quantity, unit_price,
         payment_type, country, last_updated)
) s
ON t.order_id = s.order_id
WHEN NOT MATCHED THEN INSERT *;